# Analysis of the dataset

The goal of this notebook is to gain insights about the nature of dataset used. This should help with the decision on which methodology should be consequently used.

### Module imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from collections import Counter

Data import

In [2]:
data_path = Path("../data/sportoclanky.csv.gz")

In [3]:
df = pd.read_csv(data_path)

In [4]:
df.head(10)

,category,rss_title,rss_perex
0,biatlon,"Krčmář dojel v hromadném závodě devátý, díky s...",Závod s hromadným startem v německém Oberhofu ...
1,biatlon,Česká vlajka byla v Pokljuce vidět i ve štafet...,Galerie
2,biatlon,Živě: Stíhací závod biatlonistek v Ruhpoldingu,"15. 1., 14:45"
3,fotbal,Bakoš dostal herdu do nosu a v zápase plném ka...,Slovenský útočník Marek Bakoš zařídil Plzni gó...
4,fotbal,"My moc chtěli a Plzeň moc nechtěla, zní z Ďolí...",Český fotbal hledá viníka skandálu odloženého ...
5,fotbal,Novou sezonu otevře zápas Brna s nováčkem Ostr...,Podívejte se na program prvního kola nové fotb...
6,fotbal,S Radou už Sparta moc nepočítá. Kotalík hledá ...,Pokud se ještě fotbalisté Sparty nevyhrabou na...
7,fotbal,Viktoria Žižkov musí Praze 3 zaplatit 800 tisí...,"Rozhodl o tom pražský městský soud, který potv..."
8,fotbal,Živě: Slavia získala na úvod jarní části tři b...,"18. 2., 16:45"
9,fotbal,Chvílemi jsme se báli o nohy. Ale dva góly jso...,Dvěma góly a přihrávkou nasměroval Antonín Bar...


### Dataset size

Total number of rows in the dataset

In [5]:
df_rows_count = df.shape[0]
df_rows_count

111218

### Duplicates

Fully identical rows

In [6]:
full_duplicates = sum(df.duplicated())
print(f"There is exactly {full_duplicates} of fully duplicated rows in the dataset - {full_duplicates/df_rows_count*100:.2f}% of all rows")

There is exactly 7185 of fully duplicated rows in the dataset - 6.46% of all rows


Rows where there are duplicates for title + perex

In [7]:
text_duplicates = df.duplicated(subset=["rss_title", "rss_perex"]).sum()
print(f"Duplicate texts (title+perex): {text_duplicates} ({text_duplicates/df_rows_count*100:.2f}%)")

Duplicate texts (title+perex): 7199 (6.47%)


Rows where only the titles are duplicit

In [8]:
title_duplicates = df.duplicated(subset=["rss_title"]).sum()
print(f"Duplicate titles: {title_duplicates} ({title_duplicates/df_rows_count*100:.2f}%)")

Duplicate titles: 11694 (10.51%)


Rows where same text (title + perex) are mapped to different labels over more rows

In [9]:
multi_category_duplicates = df.groupby(["rss_title", "rss_perex"])["category"].nunique()

In [10]:
conflicting = multi_category_duplicates[multi_category_duplicates > 1]
print(f"Texts assigned to > 1 category: {len(conflicting)}")

Texts assigned to > 1 category: 13


### Missing values

Function to catch both NaN as well as empty string values

In [11]:
def is_blank(col) -> bool:
    filled = col.fillna("")
    trimmed = filled.str.strip()
    return trimmed == ""
    

Apply to the datafram

In [19]:
blank = pd.DataFrame({
    "category_blank": is_blank(df["category"]),
    "title_blank": is_blank(df["rss_title"]),
    "perex_blank": is_blank(df["rss_perex"]),
})

In [20]:
blank.head()

,category_blank,title_blank,perex_blank
0,False,False,False
1,False,False,False
2,False,False,False
3,False,False,False
4,False,False,False


In [21]:
print(blank.sum())

category_blank    0
title_blank       0
perex_blank       0
dtype: int64


### Class distribution

Total 

In [14]:
counts = df["category"].value_counts()



### Text lengths